一、导入相关库

In [1]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import alphalens as al
import matplotlib.pyplot as plt
from datetime import datetime

# 配置文件
try:
    import config_local as config
except ImportError:
    import config

# 导入数据接口sdk
import zenidatasdk as zd
client = zd.Client(
    base_url=config.ZENI_URL,
    username=config.ZENI_USERNAME,
    password=config.ZENI_PASSWORD, 
)

# 忽视警告信息
warnings.filterwarnings(action='ignore')

二、获取数据

In [2]:
# 历史回测区间
init_date = '2018-01-01'
start_date = '2019-01-01'
end_date = str(datetime.today().date())
index_symbol = rf"000905.XSHG"

# rf"000300.XSHG",  # 沪深300
# rf"000905.XSHG",  # 中证500
# rf"000852.XSHG",  # 中证1000
# rf"000016.XSHG",  # 上证50
# rf"399006.XSHE",  # 创业板

In [3]:
# 获取指数成分股数据据
index_weights_df = client.get_index_constituents_df(
    index_symbol=index_symbol,
    start_date=start_date,
    end_date=end_date
)
index_weights_df = index_weights_df.rename(columns={"date": "datetime"})
symbols = index_weights_df["symbol"].unique().tolist()
index_weights_df

,datetime,index_symbol,update_date,symbol,weight,display_name
0,2019-01-02,000905.XSHG,2018-12-28,000006.XSHE,0.163,深振业A
1,2019-01-02,000905.XSHG,2018-12-28,000008.XSHE,0.291,神州高铁
2,2019-01-02,000905.XSHG,2018-12-28,000009.XSHE,0.308,中国宝安
3,2019-01-02,000905.XSHG,2018-12-28,000012.XSHE,0.148,南玻A
4,2019-01-02,000905.XSHG,2018-12-28,000021.XSHE,0.141,深科技
...,...,...,...,...,...,...
883995,2026-04-20,000905.XSHG,2026-03-31,688772.XSHG,0.110,珠海冠宇
883996,2026-04-20,000905.XSHG,2026-03-31,688777.XSHG,0.425,中控技术
883997,2026-04-20,000905.XSHG,2026-03-31,688778.XSHG,0.151,厦钨新能
883998,2026-04-20,000905.XSHG,2026-03-31,688819.XSHG,0.046,天能股份


In [4]:
# 获取日频bar数据
bars_1d_df = client.kline.get_kline_df(
    symbol=symbols,
    start_date=init_date,
    end_date=end_date,
    frequency="1d",
    adjust_type="pre",
    market="cn_stock"
)
bars_1d_df

,open,high,low,close,pre_close,high_limit,low_limit,avg,volume,datetime,symbol,amount,paused
0,8.23,8.23,8.23,8.23,8.23,8.23,8.23,8.23,0.00,2018-01-02,000006.XSHE,0.000000e+00,1.0
1,8.61,8.68,8.56,8.63,8.61,9.48,7.75,8.61,8423893.37,2018-01-02,000008.XSHE,7.252324e+07,0.0
2,5.92,5.97,5.89,5.96,5.89,6.48,5.30,5.94,23655918.27,2018-01-02,000009.XSHE,1.404975e+08,0.0
3,5.62,5.88,5.62,5.81,5.70,6.27,5.13,5.77,56707733.65,2018-01-02,000012.XSHE,3.273591e+08,0.0
4,9.18,9.22,8.99,9.16,9.06,9.97,8.16,9.14,26639867.22,2018-01-02,000021.XSHE,2.433756e+08,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1976383,67.45,69.22,66.50,68.44,67.38,80.86,53.90,68.39,14877555.00,2026-04-17,688777.XSHG,1.017431e+09,0.0
1976384,76.20,82.78,76.00,82.78,76.66,91.99,61.33,79.40,12658611.00,2026-04-17,688778.XSHG,1.005103e+09,0.0
1976385,10.86,11.13,10.82,11.05,10.92,13.10,8.74,10.98,30315926.00,2026-04-17,688779.XSHG,3.329475e+08,0.0
1976386,35.01,35.69,34.93,35.52,35.12,42.14,28.10,35.42,3514735.00,2026-04-17,688819.XSHG,1.245065e+08,0.0


In [5]:
# 获取日频估值数据
fundamental_1d_df = client.get_valuation_df(
    symbols=symbols,
    start_date=start_date,
    end_date=end_date,
    fields="datetime,symbol,market_cap,circulating_market_cap,turnover_ratio,pe_ratio,pb_ratio,dividend_ratio"
)
fundamental_1d_df


# 构建市值数据 
# market_cap、circulating_market_cap
mkt_cap_name = "market_cap"
market_cap_df = fundamental_1d_df.set_index(["datetime", "symbol"])[[mkt_cap_name]]

# 负数和无穷值 & 对数处理
market_cap_df[mkt_cap_name] = np.where((market_cap_df[mkt_cap_name] <= 0) | (~np.isfinite(market_cap_df[mkt_cap_name])), 0, market_cap_df[mkt_cap_name])
market_cap_df[f"{mkt_cap_name}_log"] = np.log1p(market_cap_df[mkt_cap_name])
market_cap = market_cap_df[f"{mkt_cap_name}_log"]
market_cap

datetime    symbol     
2019-01-02  000006.XSHE    4.255963
            000008.XSHE    4.703668
            000009.XSHE    4.537125
            000012.XSHE    4.744570
            000021.XSHE    4.437550
                             ...   
2026-04-20  688777.XSHG    0.000000
            688778.XSHG    0.000000
            688779.XSHG    0.000000
            688819.XSHG    0.000000
            689009.XSHG    0.000000
Name: market_cap_log, Length: 1764597, dtype: float64

In [6]:
# 获取行业数据
industry_constituents_df = client.get_industry_constituents_composite_df(
    symbols=symbols,
    category="sw_l1",
    start_date=start_date,
    end_date=end_date
)

# 构建双重索引的行业数据
industries = industry_constituents_df.set_index(["datetime", "symbol"])["industry_name"]
industries

datetime    symbol     
2019-01-02  000893.XSHE    农林牧渔I
            000930.XSHE    农林牧渔I
            000998.XSHE    农林牧渔I
            002041.XSHE    农林牧渔I
            002124.XSHE    农林牧渔I
                           ...  
2026-04-20  300957.XSHE    美容护理I
            600315.XSHG    美容护理I
            603605.XSHG    美容护理I
            603983.XSHG    美容护理I
            688363.XSHG    美容护理I
Name: industry_name, Length: 1785299, dtype: object

三、构建价格数据

In [7]:
# 多资产价格数据(开盘价买入)
prices_df = bars_1d_df[bars_1d_df["datetime"] >= start_date]
prices = prices_df.pivot_table(index="datetime", columns="symbol", values="open")
prices

symbol,000006.XSHE,000008.XSHE,000009.XSHE,000012.XSHE,000021.XSHE,000025.XSHE,000027.XSHE,000028.XSHE,000031.XSHE,000032.XSHE,...,688692.XSHG,688702.XSHG,688709.XSHG,688728.XSHG,688772.XSHG,688777.XSHG,688778.XSHG,688779.XSHG,688819.XSHG,689009.XSHG
datetime,,,,,,,,,,,,,,,,,,,,,
2019-01-02,4.46,3.87,3.56,3.12,5.43,18.09,3.83,28.45,4.67,6.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-03,4.44,3.85,3.50,3.13,5.37,18.04,3.75,27.63,4.64,6.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-04,4.38,3.78,3.43,3.14,5.20,18.09,3.71,27.38,4.62,6.87,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-07,4.61,3.93,3.62,3.21,5.42,18.78,3.80,27.79,4.84,7.14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-08,4.62,4.05,3.65,3.22,5.66,18.63,3.80,27.58,4.78,7.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-13,8.56,2.71,8.83,4.46,27.79,16.19,6.64,26.12,2.93,17.69,...,228.10,213.00,41.72,13.12,15.96,66.12,75.01,10.74,34.03,44.30
2026-04-14,8.62,2.75,9.00,4.55,28.80,16.27,6.71,25.58,2.98,18.29,...,236.00,230.00,43.06,13.24,16.13,65.88,74.23,11.01,34.49,44.22
2026-04-15,8.84,2.73,9.05,4.53,29.31,16.32,6.71,25.70,3.03,18.59,...,238.88,222.00,44.79,13.29,16.44,68.08,76.00,11.17,35.25,44.27


四、构建因子变量

In [ ]:
# 计算因子数据
factors_df = ... # 因子函数 func()
# 因子值 shift 1 转换成实际使用时间(T+1)
factors_df["factor_value"] = factors_df.groupby('symbol')['factor_value'].transform(lambda x: x.shift(1))
factors_df

In [ ]:
# 与指数的交易日历、历史成分股数据对齐
factors_df = pd.merge(index_weights_df[["datetime", "symbol"]], factors_df, how="left", on=["datetime", "symbol"])
# 转换成[datetime, symbol]双重索引的factor_table
factors = factors_df.pivot_table(index=["datetime", "symbol"], columns="factor_name", values="factor_value")
factors

In [ ]:
factors.info()

五、合并因子与目标变量

In [ ]:
# 调用alphalens进行数据清洗
factor_data = al.utils.get_clean_factor_and_forward_returns(
    factor=factors,
    prices=prices,
    periods=(1, 5, 20),
    bins=None,
    quantiles=5,
    groupby=None,
    groupby_labels=None,
    binning_by_group=False,
    filter_zscore=20,
    max_loss=0.25,
    zero_aware=False,
    cumulative_returns=True
)
factor_data

六、alphalens因子测试

In [ ]:
# 调用alphalens进行因子评估
al.tears.create_full_tear_sheet(
    factor_data=factor_data,
    long_short=False,
    group_neutral=False,
    by_group=False
)